# Lab - Prompt Evaluation

Three tasks that turn "this prompt looks good" into a number you can compare.

| Task | What you learn |
| --- | --- |
| 1 | An eval is a set of examples where you already know the answer |
| 2 | Running the prompt over all of them at once |
| 3 | Comparing what came back with what you expected |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

Task 1 makes no API call at all. It is reading and thinking, which is genuinely most of
what building an eval is.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

It also defines `classify_intent()` for you. That function is not the subject of this
lab - read it if you like, but you will not change it.

In [ ]:
# --- Lab setup (provided - just run it) ---
import json
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

# The five labels ShopAssist can route on. Nothing outside this list is allowed.
ALLOWED_INTENTS = [
    "refund_request",
    "order_status",
    "billing_issue",
    "product_question",
    "other",
]


# Provided. Sends one customer message to Claude and asks for the intent as JSON.
# No temperature here: anthropic SDK 1.0+ and newer Claude models no longer accept it.
# The prompt is what keeps a classification consistent.
def classify_intent(customer_message):
    if not isinstance(customer_message, str):
        raise TypeError(
            "classify_intent() needs the customer's message as text. You still have "
            "... where test_case[\"input\"] should be.")

    prompt = f"""
Classify the customer's message into one of these intents:
- refund_request
- order_status
- billing_issue
- product_question
- other

Customer message:
{customer_message}

Return only a valid JSON object.
Do not include markdown.
Do not include explanations.
Do not wrap the JSON in a code block.
{{
  "intent": "refund_request"
}}
"""
    message = client.messages.create(
        model=model,
        max_tokens=200,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )
    return json.loads(message.content[0].text)

---

## Task 1 - Build the evaluation dataset

In [ ]:
# ============================================================
# TASK 1 - Build the evaluation dataset
# ============================================================
#
# WHAT TO DO
#   Read each customer message and write down the label you
#   expect Claude to return. No code runs yet - this is you
#   deciding the right answer in advance.
#
# WHY IT MATTERS
#   This is the one thing that separates an eval from a demo. A
#   demo shows you an output and you nod at it. An eval compares
#   the output against an answer you committed to BEFORE you ran
#   anything, so you cannot talk yourself into liking a bad
#   result.
#
#   Four cases is small. Real datasets hold hundreds. The shape
#   is identical.
#
# WHERE TO SEE IT IN THE LECTURE
#   "First, we create a small evaluation dataset" - about
#   2 minutes 27 seconds in.
#
# HOW TO DO IT
#   Three blanks, one per message. Each is a label in quotes,
#   and there are only five to choose from:
#
#       "refund_request"     wants money back, or to send
#                            something back
#       "order_status"       where is my parcel
#       "billing_issue"      charged wrongly, charged twice
#       "product_question"   how does this thing work
#       "other"              none of the above
#
#   The fourth case is already filled in. Leave it as it is -
#   it is there to make a point in Task 3.
# ============================================================

test_cases = [
    # BEGIN SOLUTION
    {
        "input": "I want to return my shoes. They arrived damaged.",
        "expected_intent": "refund_request",
    },
    {
        "input": "Where is my order? It was supposed to arrive yesterday.",
        "expected_intent": "order_status",
    },
    {
        "input": "I was charged twice for the same order.",
        "expected_intent": "billing_issue",
    },
    # SCAFFOLD: {
    # SCAFFOLD:     "input": "I want to return my shoes. They arrived damaged.",
    # SCAFFOLD:     "expected_intent": ...,   # which of the five labels fits this one?
    # SCAFFOLD: },
    # SCAFFOLD: {
    # SCAFFOLD:     "input": "Where is my order? It was supposed to arrive yesterday.",
    # SCAFFOLD:     "expected_intent": ...,   # which of the five labels fits this one?
    # SCAFFOLD: },
    # SCAFFOLD: {
    # SCAFFOLD:     "input": "I was charged twice for the same order.",
    # SCAFFOLD:     "expected_intent": ...,   # which of the five labels fits this one?
    # SCAFFOLD: },
    # END SOLUTION: replace each ... below with one of the five labels, in quotes

    # Provided, and deliberately awkward - this customer has two problems at
    # once. We decided to call it a refund_request. Task 3 shows what Claude
    # decided.
    {
        "input": "I was charged twice and I want my money back.",
        "expected_intent": "refund_request",
    },
]

for case in test_cases:
    print("{:<16} <- {}".format(str(case["expected_intent"]), case["input"]))

check("eval_dataset", test_cases=test_cases)

---

## Task 2 - Run the prompt over every case

In [ ]:
# ============================================================
# TASK 2 - Run the prompt over every case
# ============================================================
#
# WHAT TO DO
#   Loop over the four test cases, send each customer message
#   through classify_intent(), and collect what came back.
#
# WHY IT MATTERS
#   One message proves nothing. The prompt that handles a clean
#   refund request may fall over on a billing dispute, and you
#   only find that out by running the whole set every time.
#
#   Notice that nothing is graded in this cell. You are looking
#   at output, which is where most people stop. Task 3 is the
#   part that makes it an eval.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now, let's run these examples through our prompt. I'll
#   create a simple loop" - about 2 minutes 48 seconds in.
#
# HOW TO DO IT
#   Two blanks, both inside the loop:
#
#       test_case["input"]            the customer's message
#       response["intent"]            the label Claude returned
#
#   classify_intent() gives back a dictionary that looks like
#   {"intent": "refund_request"}, so the label lives under the
#   "intent" key.
# ============================================================

results = []

for test_case in test_cases:
    # BEGIN SOLUTION
    response = classify_intent(test_case["input"])

    results.append({
        "input": test_case["input"],
        "expected": test_case["expected_intent"],
        "actual": response["intent"],
    })
    # SCAFFOLD: response = classify_intent(...)   # test_case["input"] - the customer's message
    # SCAFFOLD:
    # SCAFFOLD: results.append({
    # SCAFFOLD:     "input": test_case["input"],
    # SCAFFOLD:     "expected": test_case["expected_intent"],
    # SCAFFOLD:     "actual": ...,   # response["intent"] - the label Claude returned
    # SCAFFOLD: })
    # END SOLUTION: replace each ... below with the value named beside it

for row in results:
    print("expected {:<16} actual {:<16} | {}".format(
        str(row["expected"]), str(row["actual"]), row["input"][:45]))

check("eval_run", results=results)

---

## Task 3 - Compare actual with expected

In [ ]:
# ============================================================
# TASK 3 - Compare actual with expected
# ============================================================
#
# WHAT TO DO
#   For each result, decide whether the test passed, then count
#   how many did.
#
# WHY IT MATTERS
#   This is the whole idea. Change the prompt, run the same four
#   cases, and the score tells you whether the change helped -
#   instead of you deciding by feel.
#
#   Read the failing case carefully. It is not a bug. That
#   customer mentions two problems in one sentence, so both
#   labels are defensible. Which one is correct is a decision
#   your team makes and writes down; the model cannot guess it.
#   Finding disagreements like that is what evals are for.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So let's add the comparison" - about 3 minutes 33 seconds
#   in.
#
# HOW TO DO IT
#   Two blanks, both taken from the result you are looking at:
#
#       result["actual"]     what Claude answered
#       result["expected"]   what you wrote down in Task 1
#
#   Comparing two values with == gives True or False, which is
#   exactly what "passed" should hold.
# ============================================================

for result in results:
    # BEGIN SOLUTION
    actual = result["actual"]
    expected = result["expected"]
    # SCAFFOLD: actual = ...     # result["actual"] - what Claude answered
    # SCAFFOLD: expected = ...   # result["expected"] - what you wrote down in Task 1
    # END SOLUTION: replace each ... below with the value named beside it

    # Provided: catches a blank left above, so you get a message instead of a
    # comparison that is accidentally True.
    if actual is Ellipsis or expected is Ellipsis:
        raise ValueError("You still have ... above. Fill in both blanks, then run "
                         "the cell again.")

    result["passed"] = actual == expected

passed_count = sum(1 for row in results if row["passed"])

for row in results:
    print("{}  expected {:<16} actual {:<16} | {}".format(
        "PASS" if row["passed"] else "FAIL",
        str(row["expected"]), str(row["actual"]), row["input"][:42]))

print()
print("Score: {} of {}".format(passed_count, len(results)))

check("eval_compare", results=results)

---

## Done

You now have a number. That is the point.

The number itself is not meaningful - three out of four on four examples proves very
little. What is meaningful is running the *same* four cases again after you change the
prompt. If the score goes up, the change helped. If it goes down, you caught a
regression before a customer did.

Two habits worth taking from this lab. Write the expected answer before you run
anything, and always read the failures rather than just the score. The failing case
here was not a mistake by Claude - it was a question nobody on the team had answered
yet.

In the next lesson the grading gets more systematic: some things a plain `if` can check,
and some things need a second Claude call to judge.